In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
sales_raw_schema = "shop_id string, sales_amt string, date string, store_size string"
sales_raw_data = [
    {"shop_id": "1", "sales_amt": "1500.50", "date": "2026/03/07", "store_size": "Large"},
    {"shop_id": "2", "sales_amt": "invalid", "date": "2026/03/08", "store_size": ""},
    {"shop_id": "3", "sales_amt": "900.00",  "date": "2026/03/09", "store_size": None}
]

shop_raw_schema = "shop_id int, shop_name string, is_active int"
shop_raw_data = [
    {"shop_id": 1, "shop_name": "BKK-01", "is_active": 1},
    {"shop_id": 2, "shop_name": "CNX-01", "is_active": 0},
    {"shop_id": 3, "shop_name": "HKT-01", "is_active": 1}
]

pipeline_sales_df = spark.createDataFrame(sales_raw_data, schema=sales_raw_schema)
pipeline_shop_df = spark.createDataFrame(shop_raw_data, schema=shop_raw_schema)

**Step 1: Casting & Error Handling**
1. Take `pipeline_sales_df`. Cast `shop_id` to `int`.
2. Try cast `sales_amt` to `decimal(10,2)`.
3. Use `to_date` with `"yyyy/MM/dd"` to cast the `date` string.
4. Save to a new variable `sales_cast_df`. Notice what happens to the `"invalid"` text.

* **Expected Result:**
| shop_id | sales_amt | date | store_size |
|---|---|---|---|
| 1 | 1500.50 | 2026-03-07 | Large |
| 2 | null | 2026-03-08 |  |
| 3 | 900.00 | 2026-03-09 | null |

In [0]:
(
    pipeline_sales_df.withColumn("shop_id", col("shop_id").cast("int"))
    .withColumn("sales_amt", col("sales_amt").try_cast("decimal(10,2)"))
    .withColumn("date", to_date(col("date"),("yyyy/MM/dd")))  
    .display()
)

shop_id,sales_amt,date,store_size
1,1500.50,2026-03-07,Large
2,null,2026-03-08,
3,900.00,2026-03-09,null


In [0]:
sales_cast_df = (
    pipeline_sales_df.withColumn("shop_id", col("shop_id").cast("int"))
    .withColumn("sales_amt", col("sales_amt").try_cast("decimal(10,2)"))
    .withColumn("date", to_date(col("date"),("yyyy/MM/dd")))  
)
sales_cast_df.display()

shop_id,sales_amt,date,store_size
1,1500.50,2026-03-07,Large
2,null,2026-03-08,
3,900.00,2026-03-09,null


**Step 2: Filtering & Imputation (when/otherwise)**
1. Filter `sales_cast_df` to remove rows where `sales_amt` `.isNull()`.
2. Use `.withColumn` and `.when()` to replace empty strings (`""`) or `null` in `store_size` with `"Unknown"`.
3. Save as `sales_clean_df`.

* **Expected Result:**
| shop_id | sales_amt | date | store_size |
|---|---|---|---|
| 1 | 1500.50 | 2026-03-07 | Large |
| 3 | 900.00 | 2026-03-09 | Unknown |

In [0]:
 sales_clean_df = (sales_cast_df
    .where(col("sales_amt").isNotNull())
    .withColumn("store_size", when((col("store_size").isNull()) | (col("store_size") == ""), "Unknown").otherwise(col("store_size")))
  )
 sales_clean_df.display()

shop_id,sales_amt,date,store_size
1,1500.50,2026-03-07,Large
3,900.00,2026-03-09,Unknown


**Step 3: Dimension Preparation**
1. Take `pipeline_shop_df`. 
2. Cast `is_active` to `boolean`.
3. Add a hardcoded literal column `batch_year` = `2026`.
4. Save as `shop_dim_df`.

* **Expected Result:**
| shop_id | shop_name | is_active | batch_year |
|---|---|---|---|
| 1 | BKK-01 | true | 2026 |
| 2 | CNX-01 | false | 2026 |
| 3 | HKT-01 | true | 2026 |

In [0]:
pipeline_shop_df.display()

shop_id,shop_name,is_active
1,BKK-01,1
2,CNX-01,0
3,HKT-01,1


In [0]:
shop_dim_df = (pipeline_shop_df
    .withColumn("is_active", col("is_active").cast("boolean"))
    .withColumn("batch_year", lit(2026))
)
shop_dim_df.display()

shop_id,shop_name,is_active,batch_year
1,BKK-01,true,2026
2,CNX-01,false,2026
3,HKT-01,true,2026


**Step 4: Advanced GroupBy & Aggregation**
1. Take `sales_cast_df` .Replace when store_size is `""` or `null` with `"Unknown"`
1. Group `sales_cast_df` by `store_size`.
2. Calculate the count of unique `shop_id`s (`unique_stores`).
3. Calculate the sum of `sales_amt` (`total_sales`).
4. Create an array of all `date`s in that group using `collect_list()`.

* **Expected Result:**
|store_size|unique_stores|total_sales|date|
|---|---|---|---|
|Large|1|1500.50|["2026-03-07"]|
|Unknown|2|900.00|["2026-03-08","2026-03-09"]|

In [0]:
(sales_cast_df
  .withColumn("store_size", when((col("store_size") == "") | (col("store_size").isNull()), "Unknown").otherwise(col("store_size")))
  .groupBy("store_size")
  .agg(
    countDistinct("shop_id").alias("unique_stores"),
    sum("sales_amt").alias("total_sales"),
    collect_list("date").alias("dates")
    )
  .display()
)

store_size,unique_stores,total_sales,dates
Large,1,1500.50,List(2026-03-07)
Unknown,2,900.00,"List(2026-03-08, 2026-03-09)"
